# Assignment 5 - Run 1 Evaluation


===========================================================================================================

## 1. Setup

We will first install a number of libraries and import what we will need.





In [1]:
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
!pip install -q -U langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -q -U qdrant-client sentence-transformers arxiv pymupdf
!pip install -q -U transformers accelerate bitsandbytes

!pip install -q -U xmltodict

!pip install -q -U cohere

!pip install -q -U langchain-cohere

!pip install -q -U wikipedia

!pip install bert_score

In [ ]:
import os
import numpy as np
import time
import locale

# IMPORTANT: Add your Hugging Face token to Colab's Secrets (the key icon on the left panel)
# and name it 'HF_TOKEN', or replace the line below with os.environ["HF_TOKEN"] = "your_token"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

import langchain
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.utils.function_calling import convert_to_openai_tool


from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import PyMuPDFLoader

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


from langchain_cohere import ChatCohere

import torch.nn.functional as F
from torch import Tensor
from transformers import AutoModel

import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader

locale.getpreferredencoding = lambda: "UTF-8"


In [ ]:
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers.utils import get_json_schema
import json
import re
import uuid

In [ ]:
%%capture
!pip install -U sentence_transformers
from sentence_transformers import CrossEncoder


!tar -xzf "/content/drive/MyDrive/Colab Data/MIDS-267-A5/qdrant_250_50.tar.gz" -C /content

In [ ]:
%%capture

import shutil

class VectorStoreRetriever():
      EMBEDDINGS_MODEL = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
      def __init__(self):
          self.collection_name = "rag_tech_db_250"
          self.qdrant_path = '/content/qdrant_storage'
          self.init_embeddings(self.EMBEDDINGS_MODEL)
          self.init_vector_store(self.qdrant_path, self.collection_name)
      def init_vector_store(self, path, collection_name):
          self.vector_store = QdrantVectorStore(
              client=QdrantClient(path=self.qdrant_path),
              embedding=self.embeddings,
              collection_name=collection_name,
              distance=Distance.DOT)
      def init_embeddings(self, embeddings_model):
          self.embeddings = HuggingFaceEmbeddings(model_name=embeddings_model)

      def destroy(self):
          self.vector_store.client.delete_collection(self.collection_name)
          if os.path.exists(self.qdrant_path):
              shutil.rmtree(self.qdrant_path)
              print(f"Qdrant storage directory removed ({self.qdrant_path}).")

retriever = VectorStoreRetriever()
vector_store = retriever.vector_store
embedings = retriever.embeddings

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from transformers import GenerationConfig

# LOAD QWEN 3 LLM
qwen_model_name = "Qwen/Qwen3-4B-Instruct-2507"

qwen_quantization_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float32,
    device_map="auto",
    quantization_config=qwen_quantization_config
)
qwen_model.config.pad_token_id = qwen_model.config.eos_token_id



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:

qwen_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_new_tokens=1000,
    temperature=0.3,
    top_p=0.5,
    do_sample=True,
    repetition_penalty=1.2,
    pad_token_id=qwen_model.config.eos_token_id,
    eos_token_id=qwen_model.config.eos_token_id
)

qwen_llm = HuggingFacePipeline(pipeline=qwen_pipe)
qwen_chat = ChatHuggingFace(llm=qwen_llm)

In [ ]:
class EvaluatorConfig(TypedDict):
    name: str
    template: str
    generation_config: dict

In [ ]:
IDK = "[IDK]"

claims_config = EvaluatorConfig(
    name='claims',
    template="""Your task is to evaluate the claims of a provided statement based ONLY on provided context.

A claim is a single, simplified and separable clause that can be evaluated independently.
You must break the statement into its constituent claims independently of the context.

After the statement is broken into claims, classify each in exactly 1 of 3 possible categories: faithful, noisy, or irrelevant.
A claim is faithful if it is supported or verified by the supplied context. The factual basis of a faithful claim MUST appear in the context.
A claim is noisy if it is incorrect, factually wrong, or contrary to the information in the context.
Claims that are not noisy and not faithful are irrelevant - the context does not mention the information in the claim, so it cannot be determined as noisy or faithful to the context.

Context:
Frogs are green.

Statement:
Darryl is a frog, therefore he is green. Frogs are brown.

Answer:
{{
"claims": [{{ "claim":"Darryl is a frog", "evaluation":"faithful" }},
           {{ "claim": "he is green", "evaluation": "irrelevant" }},
           {{ "claim": "Frogs are brown", "evaluation": "noisy" }}],
"claims_count": 3,
"noisy_count": 1,
"faithful_count": 1,
"irrelevant_count": 1
}}

Context:
{context}

Statement:
{response}

Your response MUST be a valid JSON object enclosed in markdown tags structured with this schema:
```json
{{
"claims": array[{{ "claim":"string","evaluation":string["faithful","noisy","irrelevant"] }}],
"claims_count": integer (total count of all claims),
"noisy_count": integer (count of noisy claims),
"faithful_count": integer (count of faithful claims),
"irrelevant_count": integer (count of irrelevant claims)
}}
```""",
    generation_config={
        "max_new_tokens": 1024,
        "do_sample": False,
    }
)
persona_config = EvaluatorConfig(
    name='persona',
    template="""Your task is to evaluate how well a statement reflects the communication preferences of the user.
Some users like detailed and highly technical answers to their questions, while other users like high-level and simplified respones.

Your evaluation must be an integer in [0, 1, 2]
0 - Statement does not reflect the user's preferences
1 - Statement mostly reflects the user's preferences
2 - Statement reflects the user's preferences extremely well

Preferences:
{persona}

Statement:
{response}

Your response MUST be a valid JSON object enclosed in markdown tags structured with this schema:
```json
{{
"persona_reason": string[sentence to justify the score],
"persona_score": integer[0,1,2]
}}
```"""
,
    generation_config={
        "max_new_tokens": 1024,
        "do_sample": False,
    }
)
relevancy_config = EvaluatorConfig(
    name='relevancy',
    template="""You are test writing specialist. You will be given a statement that is the intended answer to a question.
Your task is to write a question based ONLY on the content of the statement.
When paired together, the statement should be a reasonable and accurate answer to the question.

Statement:
{response}


Provide 3 DIFFERENT questions. Write ONLY the questions. Do not provide any other text or answer to the question.
Your response MUST be a valid JSON object enclosed in markdown tags structured with this schema:
```json
{{
"questions": ["question 1","question 2","question 3"]
}}
```"""
,
    generation_config={
        "max_new_tokens": 1024,
        "top_p": 0.95,
        "temperature": 1.0,
        "do_sample": True,
    }
)


In [ ]:
import bert_score
import gc
from sklearn.metrics.pairwise import cosine_similarity

class RAGEvaluator():
    def __init__(self, tokenizer, model, embeddings, vector_store, idk, claims_config=None, persona_config=None, relevancy_config=None):
        self.tokenizer = tokenizer
        self.model = model
        self.embeddings = embeddings
        self.vector_store = vector_store
        self.idk = idk
        self.claims_config = claims_config
        self.persona_config = persona_config
        self.relevancy_config = relevancy_config
        self.faithful_scores = []
        self.noisy_scores = []
        self.persona_scores = []
        self.relevancy_scores = []
    def evaluate(self, questions, gold_answers, context_ids, responses, response_personas):
        print("Begin RAG Evaluation")
        for question, gold_answer, context, response, persona in zip(questions, gold_answers, context_ids, responses, response_personas):
            print(f"Question: {question}")
            print(f"Gold Answer: {gold_answer}")
            print(f"Response: {response}")
            print(f"Persona: {persona}")
            self.claims(response, context)
            self.persona(response, persona)
            self.relevancy(response, question)
            print("------")
        evaluation = {
            "faithfulness": np.mean(self.faithful_scores),
            "noisiness": np.mean(self.noisy_scores),
            "persona": np.mean(self.persona_scores),
            "relevancy": np.mean(self.relevancy_scores)
        }
        evaluation["hallucination"] = self.hallucination(responses, context_ids)
        evaluation["bert"] = self.bert(gold_answers, responses)
        print("*"*60)
        print(f"FULL EVALUATION")
        print("--------------------------")
        print(f"Faithfulness: {evaluation['faithfulness']:.3f}")
        print(f"Noisiness: {evaluation['noisiness']:.3f}")
        print(f"Persona: {evaluation['persona']:.3f}")
        print(f"Relevancy: {evaluation['relevancy']:.3f}")
        print(f"Hallucination: {evaluation['hallucination']['accuracy']:.3f}")
        print(f"BERT Score: {evaluation['bert']['f1']:.3f}")
        print("*"*60)
        return evaluation
    def claims(self, response: str, context_ids: str):
        context_records = self.vector_store.client.retrieve(
            ids=context_ids,
            collection_name=self.vector_store.collection_name,
            with_payload=True
        )
        context = "\n\n".join([record.payload['page_content'] for record in context_records])
        template_fill = {
            "response": response,
            "context": context[:4000]
        }
        template = self.claims_config['template']
        generation_config = self.claims_config['generation_config']
        claims_evaluation = self.invoke(template, template_fill, generation_config)
        claim_count = claims_evaluation['claims_count']
        if claim_count == 0:
            faithful = 0
            noisy = 0
        else:
            faithful_count = claims_evaluation['faithful_count']
            noisy_count = claims_evaluation['noisy_count']
            faithful = faithful_count/claim_count
            noisy = noisy_count/claim_count
        print(f"Faithful: {faithful:.3f}")
        print(f"Noisy: {noisy:.3f}")
        self.faithful_scores.append(faithful)
        self.noisy_scores.append(noisy)
        gc.collect()
        torch.cuda.empty_cache()
    def persona(self, response: str, persona: str):
        template_fill = {
            "response": response,
            "persona": persona
        }
        template = self.persona_config['template']
        generation_config = self.persona_config['generation_config']
        persona_evaluation = self.invoke(template, template_fill, generation_config)
        persona_score = persona_evaluation['persona_score']
        print(f"Persona: {persona_score:.3f}")
        self.persona_scores.append(persona_score/2)
    def relevancy(self, response: str, question: str):
        template_fill = {
            "response": response
        }
        template = self.relevancy_config['template']
        generation_config = self.relevancy_config['generation_config']
        relevancy_evaluation = self.invoke(template, template_fill, generation_config)
        embed_question = self.embeddings.embed_query(question)
        similarity_scores = []
        for n, rel_question in enumerate(relevancy_evaluation['questions']):
            embed_rel_question = self.embeddings.embed_query(rel_question)
            similarity_scores.append(cosine_similarity([embed_question], [embed_rel_question])[0][0])
        relevancy = np.mean(similarity_scores)
        print(f"Relevancy: {relevancy:.3f}")
        self.relevancy_scores.append(relevancy)
    def hallucination(self, responses: list[str], context: list[str]):
        """Measures the tool calling """
        has_context = np.array([len(c) > 0 for c in context])
        no_context = np.logical_not(has_context)
        idk = np.array([self.idk in r for r in responses])
        fact = np.logical_not(idk)
        idk_and_context = np.logical_and(idk, has_context)
        idk_and_no_context = np.logical_and(idk, no_context)
        fact_and_context = np.logical_and(fact, has_context)
        fact_and_no_context = np.logical_and(fact, no_context)
        # Accuracy = (IDK & NO Context + Response & Context) / All Responses
        accuracy = (idk_and_no_context.sum() + fact_and_context.sum()) / len(responses)
        hallucination_evaluation = {"accuracy": accuracy}
        print(f"Hallucination: {accuracy:.3f}")
        return hallucination_evaluation
    def bert(self, gold_answers: list, responses: list):
        precision, recall, f1 =  bert_score.score(responses, gold_answers, lang="en", verbose=True)
        bert_evaluation = {
            "precision": precision.mean().item(),
            "recall": recall.mean().item(),
            "f1": f1.mean().item()
        }
        print(f"BERT Score: {bert_evaluation['f1']:.3f}")
        return bert_evaluation
    def invoke(self, template, template_fill, generation_config):
        content = template.format(**template_fill)
        message = [{"role": "user", "content": content}]
        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=False,
            add_generation_prompt=True
        ).to(self.model.device)

        with torch.no_grad():
            output = self.model.generate(**input_ids,generation_config=GenerationConfig(**generation_config))

        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        del input_ids
        del output
        gc.collect()
        torch.cuda.empty_cache()

        return self.extract(decoded_output)

    def extract(self, text_output):
        assistant_tag = "assistant"
        if assistant_tag in text_output:
            response_part = text_output.split(assistant_tag, 1)[1].strip()
            if "</think>" in response_part:
                response_part = response_part.split("</think>", 1)[1].strip()
        try:
            json_output = re.search(r"```(?:json)?\s*\n(.*?)\s*\n```", response_part, re.DOTALL).group(1)
            return json.loads(json_output)
        except Exception as e:
            print(f"Extraction error, regex failed to find JSON block.\nReturning empty dictionary.")
            print(text_output)
            raise e

In [4]:
import json
import time

def load(file_name):
    file_path = f"/content/drive/MyDrive/Colab Data/MIDS-267-A5/{file_name}"

    with open(file_path, "r") as f:
        file_contents = f.read()

        data = json.loads(file_contents)
        print(f"Successfully loaded {file_path}")
        return data

def save(json_dict, file_name):
    file_path = f"/content/drive/MyDrive/Colab Data/MIDS-267-A5/{file_name}_evaluation.json"

    with open(file_path, "w") as f:
        json.dump(json_dict, f, indent=4)

    print(f"Successfully saved {file_name} to {file_path}")

In [ ]:
rag_evaluator = RAGEvaluator(
    tokenizer=qwen_tokenizer,
    model=qwen_model,
    embeddings=vector_store.embeddings,
    vector_store=vector_store,
    idk=IDK,
    claims_config = claims_config,
    persona_config = persona_config,
    relevancy_config = relevancy_config
)

file_name = "rag_eval_qas_2026-07-26 14:35:39.json"

eval_qas = load(file_name)

questions = []
gold_answers = []
context_ids = []
responses = []
response_persona = []

for qa in eval_qas:
    questions.append(qa['question'])
    gold_answers.append(qa['gold_answer'])
    responses.append(qa['response'])
    response_persona.append(qa['persona']['description'])
    context_ids.append(qa['context_metadata'])


evaluation = rag_evaluator.evaluate(questions, gold_answers, context_ids, responses, response_persona)

save(evaluation, file_name)

Successfully loaded /content/drive/MyDrive/Colab Data/MIDS-267-A5/rag_eval_qas_2026-07-26 14:35:39.json
Begin RAG Evaluation
Question: How do language model pre-training tasks contribute to the effectiveness of open-domain question answering systems?
Gold Answer: Language model pre-training tasks play a crucial role in enhancing the effectiveness of open-domain question answering systems. By leveraging tasks such as Next Sentence Prediction (NSP) and domain-specific pre-training, models like BERT can better handle noisy and irrelevant contexts, leading to improved performance. Additionally, augmenting queries with relevant contexts can significantly enhance the unsupervised machine reading capabilities of pre-trained language models, ultimately improving their ability to provide accurate and up-to-date information for dynamic knowledge-intensive tasks.
Response: This text discusses various contributions to open-domain question answering systems, focusing on two key areas: augmented lan

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 2.71 seconds, 7.37 sentences/sec
BERT Score: 0.839
************************************************************
FULL EVALUATION
--------------------------
Faithfulness: 0.460
Noisiness: 0.194
Persona: 0.550
Relevancy: 0.706
Hallucination: 1.000
BERT Score: 0.839
************************************************************
Successfully saved rag_eval_qas_2026-07-26 14:35:39.json to /content/drive/MyDrive/Colab Data/MIDS-267-A5/rag_eval_qas_2026-07-26 14:35:39.json_evaluation.json


In [7]:
import numpy as np
file_name = "rag_eval_qas_2026-07-26 14:35:39.json"
eval_qas = load(file_name)

response_len = np.mean([len(q['response']) for q in eval_qas])
print(f"Response length (avg): {response_len}")

duration = np.mean([q['duration'] for q in eval_qas])
print(f"Response time (avg): {duration}")


Successfully loaded /content/drive/MyDrive/Colab Data/MIDS-267-A5/rag_eval_qas_2026-07-26 14:35:39.json
Response length (avg): 1904.75
Response time (avg): 62.4574724685
